In [ ]:
!pip uninstall torchvision -y -q
!pip install accelerate -q
import sys
if 'torchvision' in sys.modules:
    del sys.modules['torchvision']

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter

# Load data
exec(open('training_200_FIXED.py').read())
exec(open('testing_50.py').read())

MODEL = 'roberta-base'
VALID_FALLACIES = ['Affirming the Consequent', 'Hasty Generalization', 'Appeal to Nature', 'No True Scotsman', 'Texas Sharpshooter']
label2id = {label: idx for idx, label in enumerate(VALID_FALLACIES)}
id2label = {idx: label for label, idx in label2id.items()}

print("Loading model...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(VALID_FALLACIES), id2label=id2label, label2id=label2id)
model.to(device)
print(f"✓ Model loaded on {device}\n")

# Prepare TRAINING data
print("Preparing training data...")
train_texts = [item['input'] for item in training_data]
train_labels = [label2id[item['output']['fallacy']] for item in training_data]

inputs = tokenizer(train_texts, max_length=512, truncation=True, padding=True, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}
labels = torch.tensor(train_labels).to(device)
print(f"✓ Training data ready: {len(training_data)} examples\n")

# Calculate class weights to handle imbalance
label_counts = Counter(train_labels)
total = len(train_labels)
class_weights = torch.tensor([total / (len(VALID_FALLACIES) * label_counts[i]) for i in range(len(VALID_FALLACIES))], device=device)
print(f"Class weights: {class_weights}")

print("="*70)
print("FINE-TUNING (10 epochs with weighted loss)")
print("="*70 + "\n")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)  # Lower LR
model.train()

EPOCHS = 10  # Increased
BATCH_SIZE = 8  # Larger batch
num_batches = (len(train_labels) + BATCH_SIZE - 1) // BATCH_SIZE

for epoch in range(EPOCHS):
    total_loss = 0
    for batch_idx in range(num_batches):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min(start_idx + BATCH_SIZE, len(train_labels))

        batch_inputs = {k: v[start_idx:end_idx] for k, v in inputs.items()}
        batch_labels = labels[start_idx:end_idx]

        optimizer.zero_grad()
        outputs = model(**batch_inputs, labels=batch_labels)
        loss = outputs.loss

        # Apply class weights
        logits = outputs.logits
        weighted_loss = F.cross_entropy(logits, batch_labels, weight=class_weights)

        weighted_loss.backward()
        optimizer.step()
        total_loss += weighted_loss.item()

        if (batch_idx + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS}, Batch {batch_idx+1}/{num_batches}, Loss: {weighted_loss.item():.4f}")

    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}\n")

print("✓ Fine-tuning complete!\n")

# Evaluate on TEST SET
print("="*70)
print("EVALUATING ON TEST SET (Unseen Data)")
print("="*70 + "\n")

test_texts = [item['input'] for item in test_data]
test_labels = [label2id[item['output']['fallacy']] for item in test_data]

test_inputs = tokenizer(test_texts, max_length=512, truncation=True, padding=True, return_tensors='pt')
test_inputs = {k: v.to(device) for k, v in test_inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model(**test_inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1).cpu().numpy()

y_true = [id2label[label] for label in test_labels]
y_pred = [id2label[pred] for pred in predictions]

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

print(f"Test Accuracy: {accuracy:.2%}")
print(f"Test Macro F1: {f1:.4f}\n")

print("Classification Report:\n")
print(classification_report(y_true, y_pred, labels=VALID_FALLACIES, zero_division=0))

print("\nSaving model...")
model.save_pretrained('./logical_fallacy_model')
tokenizer.save_pretrained('./logical_fallacy_model')
print("✓ Model saved!")

✓ Test data loaded: 50 examples
  - Affirming the Consequent: 10 examples
  - Hasty Generalization: 10 examples
  - Appeal to Nature: 10 examples
  - No True Scotsman: 10 examples
  - Texas Sharpshooter: 10 examples
Loading model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded on cuda

Preparing training data...
✓ Training data ready: 198 examples

Class weights: tensor([0.9900, 0.9900, 0.9900, 1.0154, 1.0154], device='cuda:0')
FINE-TUNING (10 epochs with weighted loss)

Epoch 1/10, Batch 5/25, Loss: 1.4881
Epoch 1/10, Batch 10/25, Loss: 1.7803
Epoch 1/10, Batch 15/25, Loss: 1.6178
Epoch 1/10, Batch 20/25, Loss: 1.7325
Epoch 1/10, Batch 25/25, Loss: 1.6411
Epoch 1 Average Loss: 1.7095

Epoch 2/10, Batch 5/25, Loss: 1.5019
Epoch 2/10, Batch 10/25, Loss: 1.6851
Epoch 2/10, Batch 15/25, Loss: 1.5501
Epoch 2/10, Batch 20/25, Loss: 1.7547
Epoch 2/10, Batch 25/25, Loss: 1.6083
Epoch 2 Average Loss: 1.6208

Epoch 3/10, Batch 5/25, Loss: 1.5243
Epoch 3/10, Batch 10/25, Loss: 1.6374
Epoch 3/10, Batch 15/25, Loss: 1.5749
Epoch 3/10, Batch 20/25, Loss: 1.6565
Epoch 3/10, Batch 25/25, Loss: 1.5786
Epoch 3 Average Loss: 1.6007

Epoch 4/10, Batch 5/25, Loss: 1.4911
Epoch 4/10, Batch 10/25, Loss: 1.6679
Epoch 4/10, Batch 15/25, Loss: 1.4778
Epoch 4/10, Batch

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved!


In [ ]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch

print("="*70)
print("COMPARISON: Zero-Shot RoBERTa-base vs Fine-Tuned Model")
print("="*70 + "\n")

# Load ORIGINAL RoBERTa-base (no fine-tuning)
print("Loading original RoBERTa-base (zero-shot)...\n")
base_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
base_model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=5)
base_model.to('cuda')
base_model.eval()

# Load FINE-TUNED model
print("Loading fine-tuned model...\n")
finetuned_pipe = pipeline("text-classification", model="logical_fallacy_model", device=0)

# Test examples
test_examples = [
    ("If you play cricket regularly, you're physically fit. If you're physically fit, you play cricket regularly", "Affirming the Consequent"),
    ("My friend Sudarshan is 16 years old and likes to listen to the Rolling Stones. Therefore, all 16 year old kids like listening to the Rolling Stones", "Hasty Generalization"),
    ("You've got a thing called the sun and it's always healthier than this artificial light crap", "Appeal to Nature"),
    ("No true fan of this sci-fi series liked the new movie. If you liked the movie, you're not a true fan.", "No True Scotsman"),
    ("If someone is a bad person, they will get cyberbullied online. You are getting cyberbullied online, so you must be a bad person.", "Affirming the Consequent"),
    ("When the country of Sudarshan sends its people, they're not sending their best... They're sending people that have lots of problems, and they're bringing those problems to us. They're bringing drugs. They're bringing crime. ", "Texas Sharpshooter"),
    ("One of my friends, Sudarshan, was Indian, he never ate meat. So I'm guessing that you're part Indian if you don't eat meat", "Affirming the Consequent"),
    ("If you believe in Lord Sudarshan you should never be friends with a non Sudarshanist. If you do, you're not a true Sudarshanist,", "No True Scotsman"),
    ("I met two rude people from Country Sudarshan. Everyone from X is rude.", "Hasty Generalization"),
    ("Natural is better. I tried one natural product that worked. Hence, all should work.", "Hasty Generalization")
]

VALID_FALLACIES = ['Affirming the Consequent', 'Hasty Generalization', 'Appeal to Nature', 'No True Scotsman', 'Texas Sharpshooter']

print("Testing on 7 examples:\n")
print("-"*70 + "\n")

base_correct = 0
finetuned_correct = 0

for text, expected in test_examples:
    # Zero-shot prediction (base model)
    base_inputs = base_tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to('cuda')
    with torch.no_grad():
        base_outputs = base_model(**base_inputs)
        base_logits = base_outputs.logits
        base_pred_idx = torch.argmax(base_logits, dim=1).item()
        base_pred = VALID_FALLACIES[base_pred_idx]
        base_conf = torch.softmax(base_logits, dim=1)[0][base_pred_idx].item()

    # Fine-tuned prediction
    finetuned_result = finetuned_pipe(text)
    finetuned_pred = finetuned_result[0]['label']
    finetuned_conf = finetuned_result[0]['score']

    # Check correctness
    base_correct_flag = "✅" if base_pred == expected else "❌"
    finetuned_correct_flag = "✅" if finetuned_pred == expected else "❌"

    if base_pred == expected:
        base_correct += 1
    if finetuned_pred == expected:
        finetuned_correct += 1

    print(f"Input: {text}")
    print(f"Expected: {expected}\n")
    print(f"{base_correct_flag} Zero-Shot RoBERTa:  {base_pred} (confidence: {base_conf:.2f})")
    print(f"{finetuned_correct_flag} Fine-Tuned Model: {finetuned_pred} (confidence: {finetuned_conf:.2f})\n")
    print("-"*70 + "\n")

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Zero-Shot RoBERTa:  {base_correct}/{len(test_examples)} correct = {base_correct/len(test_examples)*100:.1f}%")
print(f"Fine-Tuned Model:   {finetuned_correct}/{len(test_examples)} correct = {finetuned_correct/len(test_examples)*100:.1f}%")
print(f"\nImprovement: {(finetuned_correct - base_correct)/len(test_examples)*100:.1f} percentage points")
print("="*70)

COMPARISON: Zero-Shot RoBERTa-base vs Fine-Tuned Model

Loading original RoBERTa-base (zero-shot)...



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading fine-tuned model...



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Testing on 7 examples:

----------------------------------------------------------------------

Input: If you play cricket regularly, you're physically fit. If you're physically fit, you play cricket regularly
Expected: Affirming the Consequent

❌ Zero-Shot RoBERTa:  No True Scotsman (confidence: 0.21)
✅ Fine-Tuned Model: Affirming the Consequent (confidence: 0.96)

----------------------------------------------------------------------

Input: My friend Sudarshan is 16 years old and likes to listen to the Rolling Stones. Therefore, all 16 year old kids like listening to the Rolling Stones
Expected: Hasty Generalization

❌ Zero-Shot RoBERTa:  Texas Sharpshooter (confidence: 0.21)
✅ Fine-Tuned Model: Hasty Generalization (confidence: 0.70)

----------------------------------------------------------------------

Input: You've got a thing called the sun and it's always healthier than this artificial light crap
Expected: Appeal to Nature

❌ Zero-Shot RoBERTa:  No True Scotsman (confidence: 